# 07 · Give a project its own verb

## Context

A research team defines heat stress as temperature at or above 35 °C. That
threshold is a project assumption, not a universal CubeDynamics primitive, but
the team still wants its method to compose cleanly with the shared grammar.

## Question

How can a project package its scientific rule as a reusable verb without
modifying CubeDynamics itself?

## Analysis story

We will write a small callable factory, apply it in one readable pipe, verify
the direct and piped forms agree, and visualize both occurrence and magnitude.

In [ ]:
import xarray as xr

from cubedynamics import pipe


def heat_stress(*, threshold: float = 35.0):
    '''Return a project-owned cube → Dataset verb.'''
    # The outer function captures user configuration. CubeDynamics pipes call
    # the inner function later with the value currently moving through the pipe.
    def _op(cube: xr.DataArray) -> xr.Dataset:
        # Fail early when the incoming object cannot satisfy this method's
        # scientific contract.
        if "time" not in cube.dims:
            raise ValueError("heat_stress requires a 'time' dimension")

        # Keep occurrence (state) separate from excess heat (magnitude). A
        # Dataset lets downstream verbs choose the component they need.
        state = (cube >= threshold).rename("state")
        magnitude = (cube - threshold).where(state, 0).rename("magnitude")
        result = xr.Dataset({"state": state, "magnitude": magnitude})

        # Preserve input provenance and record the project method configuration.
        result.attrs.update(cube.attrs)
        result.attrs.update(project_verb="heat_stress", threshold=float(threshold))
        return result

    # Returning the callable—not a computed result—is what makes this a verb
    # factory compatible with pipe(cube) | heat_stress(...).
    return _op

## Prepare a small test case

A project verb needs a deterministic example that crosses the threshold at
different times and locations. This becomes both a lesson and a regression
fixture for the project's scientific contract.

In [ ]:
import numpy as np
import pandas as pd


# Build a small heat pulse with spatial offsets so state and magnitude differ
# across both time and location.
time = pd.date_range("2025-07-01", periods=10, freq="D")
y = [1, 0]
x = [0, 1, 2]
pulse = np.array([0, 1, 3, 6, 8, 5, 2, 0, -1, 1])[:, None, None]
spatial = np.array([[-1.0, 0.0, 1.0], [0.0, 1.0, 2.0]])[None, :, :]
temperature = xr.DataArray(
    31 + pulse + spatial,
    dims=("time", "y", "x"),
    coords={"time": time, "y": y, "x": x},
    name="air_temperature",
    attrs={"units": "degC", "source": "deterministic synthetic vignette"},
)
temperature

## Pipe · Apply the project rule

The project-owned verb fits the same analytical sentence as a built-in verb.
The direct call remains useful as a small equivalence test.

In [ ]:
through_pipe = (
    pipe(temperature)
    | heat_stress(threshold=35.0)
).unwrap()

# A project verb should behave identically when called directly or through a
# pipe. This assertion is the first regression test a new add-on should keep.
direct = heat_stress(threshold=35.0)(temperature)
xr.testing.assert_identical(direct, through_pipe)
through_pipe

## Figure · Audit the scientific rule

Summaries belong outside the verb unless they are part of its stated contract.
Here we derive two communication-ready views from the returned Dataset.

In [ ]:
import matplotlib.pyplot as plt

# Reduce the Dataset into two communication-ready summaries: affected area
# through time and accumulated magnitude across the study period.
daily_fraction = through_pipe["state"].mean(("y", "x"))
cumulative_magnitude = through_pipe["magnitude"].sum("time")

# Plot input, occurrence, and magnitude together so participants can audit how
# the custom scientific rule produced its outputs.
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
temperature.mean(("y", "x")).plot(ax=axes[0], marker="o", color="#8b543c")
axes[0].axhline(35, color="0.35", linestyle="--")
axes[0].set_title("Input and project threshold")
daily_fraction.plot(ax=axes[1], marker="o", color="#3f6f72")
axes[1].set_title("Derived heat-stress fraction")
axes[1].set_ylim(-0.05, 1.05)
cumulative_magnitude.plot(ax=axes[2], cmap="YlOrRd", cbar_kwargs={"label": "degree-days"})
axes[2].set_title("Derived cumulative magnitude")
plt.show()

## What the figure tells us

The threshold line explains when heat stress begins, the middle panel shows the
fraction of the study area affected each day, and the map accumulates excess
heat through time. The project rule is explicit in one verb, while the pipe
stays as small as a built-in analysis.

## Take it into a project

Move `heat_stress` into `my_project.verbs`, document why 35 °C is meaningful,
and keep the direct-versus-pipe regression test. The repository's
`examples/custom_verb_project/` directory provides a minimal package layout.